## **Preprocessing**

**Step 1: Load data raw**

In [ ]:
# Upload data hasil scraping
from google.colab import files
uploaded = files.upload()

In [ ]:
import pandas as pd

data = pd.read_excel("skincare-raw.xlsx")
data.head()

In [ ]:
# Ambil hanya kolom 'text', karena analisis sentimen fokus ke isi tweet
data = data[['text']]

# Menghapus tweet kosong (NaN)
data.dropna(inplace=True)

print("Jumlah data awal:", len(data))
data.head(15)

**Step 2: Cleaning dan case folding**

In [ ]:
import re

def clean_text (text):
  # Menghapus Username twitter
  text=re.sub(r'@[^\s]+','',text)
  # Menghapus URL/link
  text=re.sub(r'(?:@|http?://|https?://|www)\S+', '', text)
  # Menghapus simbol hashtag tetapi mempertahankan katanya
  text=re.sub(r'#', '', text)
  # Menghapus tag html
  text=re.sub('<.*?>', '', text)
  # Mengganti line baru dengan spasi
  text=re.sub("\n"," ",text)
  # Mengubah ke huruf kecil
  text=text.lower()
  # Menghapus single character
  text=re.sub(r"\b[a-zA-Z]\b", "", text)
  # Menghapus simbol/tanda baca yang tidak penting
  text=re.sub(r'["$%^&*()_+=|~`{}\[\]:;<>/\\]', ' ', text)
  # Merapikan spasi pada teks
  text=' '.join(text.split())
  return text

data['clean_text'] = data['text'].apply(clean_text)
data.head()

**Step 3: Filtering**

In [ ]:
# Menghapus tweet duplikat
data.drop_duplicates(subset='clean_text', inplace=True)

print("Jumlah data setelah filtering duplikat data:", len(data))

In [ ]:
# Hanya ambil tweet yang punya ≥ 3 kata
data = data[data['clean_text'].str.split().str.len() >= 3]

print("Jumlah data setelah filtering 3 kata:", len(data))

**Step 4: Tokenizing**

In [ ]:
import nltk
nltk.download('punkt_tab')
from nltk.tokenize import word_tokenize

data['tokenized_text'] = data['clean_text'].apply(word_tokenize)
data.head()

**Step 5: Normalizing**

In [ ]:
def normalisasi1_text(list_kata):
  # Mengubah kata berulang ("bagussss" menjadi "bagus")
  return [re.sub(r'(.)\1{2,}', r'\1', kata) for kata in list_kata]

data['normalized_text'] = data['tokenized_text'].apply(normalisasi1_text)
data.head()

In [ ]:
# Colloquial Indonesian Lexicon (2018)
kamus_alay = pd.read_csv('https://raw.githubusercontent.com/nasalsabila/kamus-alay/master/colloquial-indonesian-lexicon.csv')
kamus_alay = kamus_alay.filter(['slang', 'formal'], axis=1)
kamus_alay = kamus_alay.drop_duplicates(subset=['slang'], keep='first')
kamus_alay = kamus_alay.set_index('slang')

kamus_dict = kamus_alay['formal'].to_dict()

In [ ]:
# Penambahan kamus secara mandiri
custom_dict = {
    'co': 'checkout',
    'po': 'preorder',
    'bo': 'breakout',
    'flashsale': 'flash sale',
    'soldout': 'sold out',
    'restok': 'restock',
    'teksturx': 'teksturnya',
    'wangy': 'wangi',
    'lembabin': 'melembapkan',
    'ngelembabin': 'melembapkan',
    'nyerep': 'menyerap',
    'berlu': 'perlu',
    'st': 'skin type',
    'skintype': 'skin type',
    'fw': 'face wash',
    'moist': 'moisturizer',
    'mw': 'micellar water',
    'skinker': 'skincare',
    'mehong': 'mahal',
    'emejing': 'amazing',
    'mantul': 'mantap betul',
    'kalo': 'kalau',
    'tbh': 'to be honest'
}

# Menggabungkan dengan kamus CIL
kamus_final = {**kamus_dict, **custom_dict}

In [ ]:
def normalisasi2_text(list_kata):
    return [kamus_final.get(word, word) for word in list_kata]

data['normalized_text'] = data['normalized_text'].apply(normalisasi2_text)

# Menggabungkan kembali list kata menjadi satu kalimat utuh (string)
data['normalized_text'] = data['normalized_text'].apply(lambda x: ' '.join(x))
data.head()

**Step 6: Simpan hasil preprocessing**

In [ ]:
# Menyimpan hasil preprocessing data raw dalam format excel untuk diberi label manual
data.to_excel('skincare-cleaned.xlsx', index=False)

## **Cohen's Kappa**

In [ ]:
# Upload data yang sudah diberi label secara manual oleh 2 annotators
from google.colab import files
uploaded = files.upload()

In [ ]:
import pandas as pd
from sklearn.metrics import cohen_kappa_score

annotator1 = pd.read_excel('skincare-labeled1.xlsx')
annotator2 = pd.read_excel('skincare-labeled2.xlsx')

label_a1 = annotator1['label']
label_a2 = annotator2['label']

# Hitung skor Kappa
kappa_score = cohen_kappa_score(label_a1, label_a2)

print(f"Nilai Cohen's Kappa: {kappa_score:.4f}")

In [ ]:
import matplotlib.pyplot as plt

label_map = {0: 'Positive', 1: 'Neutral', 2: 'Negative'}

# 1. Tampilkan jumlah angkanya dulu
print(annotator1['label'].map(label_map).value_counts())

# 2. Baru buat dan tampilkan grafiknya
annotator1['label'].map(label_map).value_counts().plot(
    kind='pie',
    autopct='%1.1f%%',
    title='Distribusi Sentimen Multikelas (Annotator 1)'
)
plt.ylabel('')
plt.show()

In [ ]:
import matplotlib.pyplot as plt

label_map = {0: 'Positive', 1: 'Neutral', 2: 'Negative'}

# 1. Tampilkan jumlah angkanya dulu
print(annotator2['label'].map(label_map).value_counts())

# 2. Baru buat dan tampilkan grafiknya
annotator2['label'].map(label_map).value_counts().plot(
    kind='pie',
    autopct='%1.1f%%',
    title='Distribusi Sentimen Multikelas (Annotator 2)'
)
plt.ylabel('')
plt.show()

## **Exploratory Data Analysis**

In [ ]:
# Upload data yang sudah diberi label
from google.colab import files
uploaded = files.upload()

In [ ]:
import pandas as pd

data = pd.read_excel('skincare-labeled.xlsx')

label = data['label']

In [ ]:
import matplotlib.pyplot as plt

label_map = {0: 'Positive', 1: 'Neutral', 2: 'Negative'}

# 1. Tampilkan jumlah angkanya dulu
print(data['label'].map(label_map).value_counts())

# 2. Baru buat dan tampilkan grafiknya
data['label'].map(label_map).value_counts().plot(
    kind='pie',
    autopct='%1.1f%%',
    title='Distribusi Sentimen Multikelas Pelabelan Manual'
)
plt.ylabel('')
plt.show()

In [ ]:
import matplotlib.pyplot as plt

label_map = {0: 'Positive', 1: 'Neutral', 2: 'Negative'}

# 1. Tampilkan jumlah angkanya dulu
print(data['label'].map(label_map).value_counts())

# 2. Baru buat dan tampilkan grafiknya
data['label'].map(label_map).value_counts().plot(
    kind='bar',
    title='Distribusi Sentimen Multikelas Pelabelan Manual',
    color=['#1f77b4', '#ff7f0e', '#2ca02c'],
    rot=0
)
plt.ylabel('')
plt.show()